# GLIDE Footprint Explorer

Interactive map of LPDM residence-time footprints written to a `footprints.zarr` store. Renders one altitude bin at a time as a true raster (`hvplot.image`) on a tile basemap, with a slider for the time-ago dimension.

Scales cleanly from the local SF demo (40×40) to the FLEXPART-aligned EUROPE grid (391×293) and beyond — each animation frame is a single raster image, not 100k+ polygons. Change `RUN_DIR` to point at any GLIDE output directory.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import xarray as xr

import holoviews as hv
import geoviews as gv
import hvplot.xarray  # noqa: F401  registers the .hvplot accessor on DataArray

hv.extension("bokeh")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RUN_DIR = PROJECT_ROOT / "outputs" / "comparison-mhd-202401-day01"
FOOTPRINT_ZARR = RUN_DIR / "footprints.zarr"
RUN_METADATA_JSON = RUN_DIR / "run_metadata.json"

print(f"Run dir: {RUN_DIR}")
print(f"Footprint store exists: {FOOTPRINT_ZARR.exists()}")
print(f"Run metadata exists:    {RUN_METADATA_JSON.exists()}")

## Load the footprint store and metadata

The footprint is small enough (a few tens of MB) that we eagerly `.load()` it into memory. Release coordinates come from `run_metadata.json`.

In [ ]:
footprint_ds = xr.open_zarr(FOOTPRINT_ZARR).load()
footprint = footprint_ds["footprint"]

metadata = json.loads(RUN_METADATA_JSON.read_text(encoding="utf-8"))
release_lon = float(metadata["config"]["release"]["lon"])
release_lat = float(metadata["config"]["release"]["lat"])

n_time, n_z, n_y, n_x = footprint.shape
print(f"footprint dims: time_ago={n_time}, z_bin={n_z}, lat={n_y}, lon={n_x}")
print(f"release point:  ({release_lon:.4f}, {release_lat:.4f})")
print(f"altitude bins:  {list(zip(footprint_ds['z_bottom_m'].values.tolist(), footprint_ds['z_top_m'].values.tolist()))}")

## Animated map: one altitude bin

The widget shows the footprint at one altitude bin, with a slider scrubbing through the time-ago axis. Pan, zoom, and hover work directly. Change `LEVEL_INDEX` to view a different altitude bin.

In [ ]:
LEVEL_INDEX = 0

level_da = footprint.isel(z_bin=LEVEL_INDEX)
# Mask zeros to NaN so the log color scale is valid and empty cells stay transparent.
level_da_positive = level_da.where(level_da > 0)

if not np.isfinite(level_da_positive.values).any():
    print(f"No nonzero cells at z_bin {LEVEL_INDEX}. Try a different LEVEL_INDEX.")
    plot = None
else:
    vmin = float(np.nanmin(level_da_positive.values))
    vmax = float(np.nanmax(level_da_positive.values))
    z_bottom = float(footprint_ds["z_bottom_m"].values[LEVEL_INDEX])
    z_top = float(footprint_ds["z_top_m"].values[LEVEL_INDEX])

    raster = level_da_positive.hvplot.image(
        x="longitude",
        y="latitude",
        groupby="time_ago",
        tiles="CartoLight",
        cmap="magma",
        logz=True,
        clim=(vmin, vmax),
        geo=True,
        project=False,
        height=600,
        width=900,
        alpha=0.85,
        title=f"Footprint at {z_bottom:.0f}-{z_top:.0f} m AGL",
    )
    release_marker = gv.Points([(release_lon, release_lat)]).opts(
        color="cyan",
        size=10,
        line_color="black",
        tools=["hover"],
    )
    plot = raster * release_marker

plot

## Time-integrated footprint

Same altitude bin (`LEVEL_INDEX`), but collapsed over `time_ago` into a single static map. This is the total residence-time sensitivity over the whole backward integration window — the quantity FLEXPART / NAME / STILT comparisons typically use.

In [ ]:
integrated = level_da.sum(dim="time_ago")
integrated_positive = integrated.where(integrated > 0)

if not np.isfinite(integrated_positive.values).any():
    print(f"No nonzero cells at z_bin {LEVEL_INDEX} after time integration.")
    integrated_plot = None
else:
    vmin_int = float(np.nanmin(integrated_positive.values))
    vmax_int = float(np.nanmax(integrated_positive.values))

    integrated_raster = integrated_positive.hvplot.image(
        x="longitude",
        y="latitude",
        tiles="CartoLight",
        cmap="magma",
        logz=True,
        clim=(vmin_int, vmax_int),
        geo=True,
        project=False,
        height=600,
        width=900,
        alpha=0.85,
        title=f"Time-integrated footprint, {z_bottom:.0f}-{z_top:.0f} m AGL",
    )
    integrated_plot = integrated_raster * release_marker

integrated_plot